This is where RAG systems go from “works sometimes” to consistently accurate.

Let’s walk through reranking in the same structured, intuitive way:

👉 What

👉 Why

👉 Core idea

👉 Each technique

👉 Deep analogies

👉 When to use

🧠 1. What is Reranking?

🔍 Simple Definition

👉 Reranking = reordering retrieved documents based on deeper relevance

Top-K Retrieved Docs → Rerank → Top-N Best Docs → LLM Answer

🔄 Where It Fits

User Query

   ↓

Retrieval (BM25 / Vector / Hybrid)

   ↓

Reranking ⭐

   ↓

Final Context

   ↓

LLM Answer

❗ Problem It Solves

Retrieval gives:

Top 5 docs ≠ Best 5 docs

👉 Some are:

loosely relevant

noisy

misleading

🧠 Analogy (Restaurant Search)

You search:

“Best pizza near me”

Google shows:

Pizza place ✔

Italian restaurant ✔

Bakery ❌

Fast food ❌

👉 Retrieval = initial list

👉 Reranking = sorting by actual pizza quality

🔥 2. Why Reranking is Critical

👉 Retrieval optimizes recall (find many possibilities)

👉 Reranking optimizes precision (pick the best ones)

🟣 3. Types of Reranking Techniques

We’ll cover:

LLM Score-based

Pairwise

Listwise ⭐

Query-aware

Hybrid

🔵 4. LLM Score-Based Reranking

🔍 What it does

👉 Assign score (0–1) to each document

🧠 Analogy (Teacher Grading Papers)

Teacher evaluates each answer:

Doc A → 0.9
Doc B → 0.6
Doc C → 0.3

👉 Sort by score → best first

🧪 Example

Query:

"How to reduce investment risk?"

Docs:

A: Diversification reduces risk ✔ (0.95)

B: Stock prices fluctuate ❌ (0.4)

C: Portfolio balancing ✔ (0.85)

👉 Final ranking:

A > C > B

✅ Pros

Simple

Easy to implement

❌ Cons

Scores may be inconsistent

Depends on prompt quality

🟡 5. Pairwise Reranking ⭐

🔍 What it does

👉 Compare two documents at a time

🧠 Analogy (Tournament)

Like IPL knockout:

A vs B → A wins

A vs C → C wins

C vs B → C wins

👉 Final ranking based on wins

🧪 Example

Query:

"neural networks"

Docs:

A: Neural networks basics

B: Cloud computing

C: Deep learning models

Comparisons:

A vs B → A

A vs C → C

C vs B → C

👉 Ranking:

C > A > B

✅ Pros

Very accurate

Strong reasoning

❌ Cons

Slow (O(n²))

Expensive

🔵 6. Listwise Reranking ⭐ (Best Modern Approach)

🔍 What it does

👉 LLM ranks all documents together

🧠 Analogy (Judge Ranking Contestants)

Judge sees all participants at once:

1st → C

2nd → A

3rd → B

👉 Better than comparing one-by-one

🧪 Example

Query:

"cloud computing benefits"

Docs:

A: scalable infrastructure ✔

B: stock market ❌

C: distributed systems ✔

LLM outputs:

[0, 2, 1]

👉 Ranking:

A > C > B

✅ Pros

Best balance of speed + quality

Global understanding

❌ Cons

Slightly complex prompt

🟢 7. Query-Aware Reranking

🔍 What it does

👉 Extract intent first, then rank

🧠 Analogy (Doctor Diagnosis)

Patient says:

“I feel weak”

Doctor extracts intent:

Possible anemia / fatigue

Then evaluates reports accordingly.

🧪 Example

Query:

"why investments fail?"

Intent:

financial risk causes

Docs:

A: diversification ✔

B: market fluctuations ✔

C: unrelated tech doc ❌

👉 Ranked using intent awareness

✅ Pros

Handles ambiguous queries

More contextual

❌ Cons

Extra LLM call

🔴 8. Hybrid Reranking

🔍 What it does

👉 Combine:

LLM score

Vector similarity

Keyword match

🧠 Analogy (College Admission)

Final score =

0.5 * Exam Score + 0.3 * Interview + 0.2 * Projects

👉 Not relying on one signal

🧪 Example

Doc scores:

Doc	    Vector	LLM	  Final

A	    0.8	    0.9	  0.85

B	    0.9	    0.5	  0.7

👉 A wins (better overall)

✅ Pros

Most robust

Production-grade

❌ Cons

More complex

🔥 9. Deep Analogy (Best Way to Remember)

🎓 Hiring Process

Step 1: Resume Screening → Retrieval

Step 2: Evaluation → Reranking

Different Techniques:

Technique	        Hiring Equivalent

Score-based	        Assign marks to resumes

Pairwise	        Compare candidates head-to-head

Listwise	        Rank all candidates together

Query-aware	        Match job role intent

Hybrid	            Combine marks + interview

🧠 10. When to Use What

Scenario	             Technique

Simple system	         Score-based

Small dataset	         Pairwise

Production RAG	         Listwise ⭐

Ambiguous queries	     Query-aware

Enterprise system	     Hybrid ⭐

🔄 11. Full RAG Pipeline (Modern)

User Query

   ↓

Intent Validation

    |

Query Formulation

   ↓

Query Expansion

   ↓

Prefilter
   
   |

Hybrid Retrieval (BM25 + Vector)

   ↓

RRF Fusion

   ↓

Reranking ⭐

   ↓

Top-K Context

   ↓

LLM Answer

🎯 Final Summary

Technique	          Core Idea

Score-based	          Assign score

Pairwise	          Compare pairs

Listwise	          Rank all together

Query-aware	          Use intent

Hybrid	              Combine signals

🧠 Final Insight (Very Important)

👉 Retrieval answers:

"What could be relevant?"

👉 Reranking answers:

"What is MOST relevant?"

In [1]:
import os
import json
import re
import chromadb
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI

load_dotenv()
assert os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
collection = client.create_collection("rerank_demo")

documents = [
    "Diabetes affects blood sugar levels.",
    "Hypertension increases heart disease risk.",
    "Diversification reduces investment risk.",
    "Neural networks are used in deep learning.",
    "Cloud computing provides scalable infrastructure."
]

collection.add(
    documents=documents,
    embeddings=[embedding_model.encode(d).tolist() for d in documents],
    ids=[str(i) for i in range(len(documents))]
)

In [2]:
def retrieve(query, k=5):
    emb = embedding_model.encode(query).tolist()
    results = collection.query(query_embeddings=[emb], n_results=k)
    return results["documents"][0]

LLM Score-Based Reranking

In [ ]:
# 🔍 1. Purpose of This Function

# 👉 Extract a numeric score (float) from a string

# Why needed?

# LLMs often return messy outputs like:

# "Score: 0.85 because it is relevant"

# 👉 You only need:

# 0.85

# 🔧 2. Line-by-Line Explanation

# 🔹 Step 1: Function Definition

# def extract_score(text):

# 👉 Input:

# text → string (LLM response)

# Example:

# text = "Score: 0.85 because it is relevant"

# 🔹 Step 2: Regex Search

# match = re.search(r"\d*\.?\d+", text)

# What is happening?

# 👉 Searching for the first number inside the text

# 🔍 Understanding the Regex Pattern

# r"\d*\.?\d+"

# Break it down:

# Part	Meaning
# \d*	0 or more digits
# \.?	optional decimal point
# \d+	at least one digit

# ✅ Matches Examples

# Input	               Match
# "0.85"	           0.85
# "Score: 1"	        1
# "0.5 relevance"	   0.5
# "Score = 10"	       10

def extract_score(text):
    match = re.search(r"\d*\.?\d+", text)
    return float(match.group()) if match else 0.0


def rerank_score(query, docs):
    scored = []

    for doc in docs:
        prompt = f"""
        Score relevance from 0 to 1.
        Return only number.

        Query: {query}
        Document: {doc}
        """

        score = extract_score(llm.invoke(prompt).content)
        scored.append((doc, score))

    return [d for d, _ in sorted(scored, key=lambda x: x[1], reverse=True)]

Pairwise Reranking

Compare two docs → choose better

In [4]:
def pairwise_compare(query, doc1, doc2):
    prompt = f"""
    Which document is more relevant?

    Query: {query}

    A: {doc1}
    B: {doc2}

    Answer A or B.
    """

    return llm.invoke(prompt).content.strip()


def rerank_pairwise(query, docs):
    scores = {doc: 0 for doc in docs}

    for i in range(len(docs)):
        for j in range(i + 1, len(docs)):
            result = pairwise_compare(query, docs[i], docs[j])

            if "A" in result:
                scores[docs[i]] += 1
            else:
                scores[docs[j]] += 1

    return sorted(scores, key=scores.get, reverse=True)

Listwise Reranking

LLM ranks ALL docs at once

In [5]:
def rerank_listwise(query, docs):
    docs_text = "\n".join([f"{i}: {d}" for i, d in enumerate(docs)])

    prompt = f"""
    Rank documents by relevance.

    Query: {query}

    Documents:
    {docs_text}

    Return ordered indices as JSON list.
    """

    response = llm.invoke(prompt).content

    try:
        order = json.loads(response)
        return [docs[i] for i in order]
    except:
        return docs

Query-Aware Reranking

Extract intent → rerank accordingly

In [6]:
def extract_intent(query):
    prompt = f"""
    Extract intent (short phrase).

    Query: {query}
    """
    return llm.invoke(prompt).content


def rerank_query_aware(query, docs):
    intent = extract_intent(query)

    scored = []
    for doc in docs:
        prompt = f"""
        Score relevance (0-1)

        Intent: {intent}
        Document: {doc}
        """
        score = extract_score(llm.invoke(prompt).content)
        scored.append((doc, score))

    return [d for d, _ in sorted(scored, key=lambda x: x[1], reverse=True)]

Hybrid Reranking

Combine vector similarity + LLM

🔍 1. Function Definition

def rerank_hybrid(query, docs):

👉 Inputs:

query → user question

docs → list of retrieved documents

Example:

query = "How to reduce investment risk?"

docs = ["Diversification reduces risk", "Stock markets fluctuate"]

🔹 2. Query Embedding

q_emb = embedding_model.encode(query)

👉 Converts query → vector (numerical representation)

🧠 Why?

Computers can’t understand text directly, so:

"investment risk" → [0.12, -0.45, 0.89, ...]

🧠 Analogy

Think of this like converting a sentence into coordinates on a map.

🔹 3. Initialize Score List

scored = []

👉 This will store:

[(doc1, score1), (doc2, score2)]

🔹 4. Loop Through Documents

for doc in docs:

👉 Process each document one by one

🔹 5. Document Embedding

d_emb = embedding_model.encode(doc)

👉 Convert document → vector

🔹 6. Cosine Similarity (VERY IMPORTANT)

sim = np.dot(q_emb, d_emb) / (np.linalg.norm(q_emb) * np.linalg.norm(d_emb))

🔍 What is this?

👉 Computes cosine similarity between query and document

🧠 Meaning

Measures how similar two vectors are

📊 Output Range

-1 → completely opposite

 0 → unrelated

 1 → identical meaning

🧠 Analogy (Compass Direction)

Imagine:

Query = North direction

Doc = another direction

👉 If both point same way → similarity = high

👉 If opposite → similarity = low

🔹 7. LLM Prompt

prompt = f"""

Score relevance (0-1)

Query: {query}

Document: {doc}
"""

👉 Asking LLM:

“How relevant is this document to the query?”

🧠 Why use LLM?

Embedding similarity is:

fast

but shallow

LLM is:

slower

but deep reasoning

🔹 8. LLM Scoring

llm_score = extract_score(llm.invoke(prompt).content)

🔍 What happens here?

Send prompt to LLM

LLM returns text:

"0.85 because it's relevant"

extract_score() → extracts:

0.85

🧠 Insight

👉 You now have two signals:

sim → mathematical similarity

llm_score → reasoning-based relevance

🔹 9. Combine Scores (Hybrid Magic ⭐)

final_score = 0.5 * sim + 0.5 * llm_score

🔍 What is happening?

👉 Weighted average of:

vector similarity

LLM score

🧠 Analogy (College Admission)

Final score =

50% entrance exam + 50% interview

🧪 Example

Metric	Value

sim	0.8

llm_score	0.6

👉 Final:

0.5*0.8 + 0.5*0.6 = 0.7

🔥 Why this is powerful

Embeddings → fast & consistent

LLM → smart & contextual

👉 Together = best of both worlds

🔹 10. Store Result

scored.append((doc, final_score))

👉 Save:

("Diversification reduces risk", 0.9)

🔹 11. Sort Documents

sorted(scored, key=lambda x: x[1], reverse=True)

🔍 What this does

👉 Sort by final_score (highest first)

Example

Before:

[("A", 0.6), ("B", 0.9), ("C", 0.7)]

After:

[("B", 0.9), ("C", 0.7), ("A", 0.6)]

🔹 12. Return Only Documents

[d for d, _ in ...]


👉 Remove scores, keep only documents

Final Output

["Best doc", "Second best", "Third best"]

🧠 Full Flow (Big Picture)

Query

  ↓

Embedding similarity (fast)

  ↓

LLM reasoning (deep)

  ↓

Combine scores

  ↓

Sort

  ↓

Top documents



In [7]:
def rerank_hybrid(query, docs):
    q_emb = embedding_model.encode(query)

    scored = []
    for doc in docs:
        d_emb = embedding_model.encode(doc)

        sim = np.dot(q_emb, d_emb) / (np.linalg.norm(q_emb) * np.linalg.norm(d_emb))

        prompt = f"""
        Score relevance (0-1)

        Query: {query}
        Document: {doc}
        """

        llm_score = extract_score(llm.invoke(prompt).content)

        final_score = 0.5 * sim + 0.5 * llm_score
        scored.append((doc, final_score))

    return [d for d, _ in sorted(scored, key=lambda x: x[1], reverse=True)]

FULL PIPELINE (END-TO-END)

In [8]:
def rag_with_reranking(query, method="listwise"):

    # Step 1: Retrieval
    docs = retrieve(query)
    print("📄 Retrieved:", docs)

    # Step 2: Reranking
    if method == "score":
        ranked = rerank_score(query, docs)
    elif method == "pairwise":
        ranked = rerank_pairwise(query, docs)
    elif method == "listwise":
        ranked = rerank_listwise(query, docs)
    elif method == "query":
        ranked = rerank_query_aware(query, docs)
    else:
        ranked = rerank_hybrid(query, docs)

    print("⭐ Reranked:", ranked)

    # Step 3: Generation
    context = "\n".join(ranked[:3])

    prompt = f"""
    Answer using context only.

    Context:
    {context}

    Query:
    {query}
    """

    answer = llm.invoke(prompt).content

    return {
        "query": query,
        "top_docs": ranked[:3],
        "answer": answer
    }

In [9]:
if __name__ == "__main__":
    result = rag_with_reranking(
        "How to reduce investment risk?",
        method="listwise"
    )

    import json
    print(json.dumps(result, indent=2))

📄 Retrieved: ['Diversification reduces investment risk.', 'Hypertension increases heart disease risk.', 'Cloud computing provides scalable infrastructure.', 'Neural networks are used in deep learning.', 'Diabetes affects blood sugar levels.']
⭐ Reranked: ['Diversification reduces investment risk.', 'Hypertension increases heart disease risk.', 'Cloud computing provides scalable infrastructure.', 'Neural networks are used in deep learning.', 'Diabetes affects blood sugar levels.']
{
  "query": "How to reduce investment risk?",
  "top_docs": [
    "Diversification reduces investment risk.",
    "Hypertension increases heart disease risk.",
    "Cloud computing provides scalable infrastructure."
  ],
  "answer": "Diversification reduces investment risk."
}


CrossEncoder Reranking

In [ ]:
import numpy as np
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
load_dotenv()

# -----------------------------
# 1. Initialize Models
# -----------------------------
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [11]:
# -----------------------------
# 2. Knowledge Base
# -----------------------------
documents = [
    "To open a savings account, you need Aadhaar, PAN card, and address proof.",
    "Savings account interest rates vary between 2% and 6%.",
    "Personal loans can be applied online with minimal documentation.",
    "KYC documents include identity proof, address proof, and photographs.",
    "Fixed deposits offer higher returns than savings accounts."
]

In [12]:
# -----------------------------
# 3. Query
# -----------------------------
query = "What documents are required to open a savings account?"


In [13]:
# -----------------------------
# 4. Embedding Retrieval
# -----------------------------
doc_embeddings = embedding_model.embed_documents(documents)
query_embedding = embedding_model.embed_query(query)

# cosine similarity
scores = [np.dot(doc_emb, query_embedding) for doc_emb in doc_embeddings]

# Top-K retrieval
top_k = 5
top_k_idx = np.argsort(scores)[-top_k:][::-1]
retrieved_docs = [documents[i] for i in top_k_idx]

print("\n🔹 Retrieved Docs:")
for doc in retrieved_docs:
    print("-", doc)


🔹 Retrieved Docs:
- To open a savings account, you need Aadhaar, PAN card, and address proof.
- Savings account interest rates vary between 2% and 6%.
- KYC documents include identity proof, address proof, and photographs.
- Fixed deposits offer higher returns than savings accounts.
- Personal loans can be applied online with minimal documentation.


In [14]:
# -----------------------------
# 5. 🔥 Cross-Encoder (LLM Scoring per doc)
# -----------------------------
def score_doc(query, doc):
    prompt = f"""
You are a relevance scoring system.

Score the relevance of the document to the query between 0 and 1.

Rules:
1.0 = exact answer
0.7 = highly relevant
0.4 = partially relevant
0.0 = irrelevant

Return ONLY the score.

Query:
{query}

Document:
{doc}
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    score_text = response.content.strip()

    try:
        return float(score_text)
    except:
        return 0.0

doc_scores = []
for doc in retrieved_docs:
    score = score_doc(query, doc)
    doc_scores.append((doc, score))

# Sort by score
reranked_docs = sorted(doc_scores, key=lambda x: x[1], reverse=True)

print("\n🔥 Reranked Docs:")
for doc, score in reranked_docs:
    print(f"{score:.2f} -> {doc}")


🔥 Reranked Docs:
1.00 -> To open a savings account, you need Aadhaar, PAN card, and address proof.
0.40 -> KYC documents include identity proof, address proof, and photographs.
0.00 -> Savings account interest rates vary between 2% and 6%.
0.00 -> Fixed deposits offer higher returns than savings accounts.
0.00 -> Personal loans can be applied online with minimal documentation.


In [15]:
# -----------------------------
# 6. Select Top-N
# -----------------------------
top_n_docs = [doc for doc, _ in reranked_docs[:2]]

In [16]:
# -----------------------------
# 7. Final Answer Generation
# -----------------------------
context = "\n".join(top_n_docs)

final_prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{query}
"""

response = llm.invoke([HumanMessage(content=final_prompt)])

print("\n✅ Final Answer:\n")
print(response.content)


✅ Final Answer:

To open a savings account, you need Aadhaar, PAN card, and address proof.


🔥 Better Version (Batch Cross-Encoder — Recommended)

Instead of calling LLM multiple times, do true production batching:

In [17]:
def batch_rerank(query, docs):
    doc_text = "\n\n".join([f"{i+1}. {doc}" for i, doc in enumerate(docs)])

    prompt = f"""
You are a ranking system.

Score each document between 0 and 1 based on relevance.

Return JSON format:
[
  {{"doc": 1, "score": 0.95}},
  {{"doc": 2, "score": 0.2}}
]

Query:
{query}

Documents:
{doc_text}
"""

    response = llm.invoke([HumanMessage(content=prompt)])

    return response.content